# CPE15 MIDTERM PROJECT - Cleaning Notebook
---
## Part I: Field Data Collection, Validation, Cleaning, and Documentation
---
### BSCPE 3GF
### Submitted on: Oct 11, 2026




In [ ]:
import numpy as np
import pandas as pd

## Function Definitions

In [ ]:
# Canonical schema shared by every raw CSV, plus a loader that enforces it
# The 16 required columns, in the order required by the spec
columns = [
    'feature_id', 'name', 'category', 'subcategory',
    'street', 'barangay', 'latitude', 'longitude',
    'location_description', 'operating_status', 'opening_hours',
    'contact_info', 'date_collected', 'collector',
    'remarks', 'verified'
]

# SW Maps GPS/telemetry columns, kept for the QA checks in section 10
# instead of being dropped at load time
qa_columns = [
    'Time', 'Horizontal Accuracy', 'HDOP',
    'Satellites in Use', 'Fix ID', 'Averaged Count'
]

# Non-spec raw headers that are merged into the schema or kept for traceability
# (Latitude/Longitude are renamed; Remarks is merged; Image/establishment_pic
# hold the photo reference in files 9 and 10)
raw_only_columns = ['Latitude', 'Longitude', 'Remarks', 'Image', 'establishment_pic']

# Everything the loader is allowed to pull out of each raw CSV
read_columns = set(columns) | set(qa_columns) | set(raw_only_columns)

# Final column order of the merged raw DataFrame
output_columns = columns + ['source_file', 'establishment_pic'] + qa_columns


def read_csv_files(x):
  """Read one raw CSV as text and return the spec columns, GPS QA columns, and traceability columns."""

  # dtype=str keeps phone numbers, leading zeros, and IDs exactly as written
  # (without it, e.g. 09602081305 in files 2 and 4 becomes 9602081305.0)
  df = pd.read_csv(x, dtype=str, usecols=lambda c: c in read_columns)

  # SW Maps exports use capitalized coordinate headers
  df = df.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})

  # Merge the stray capitalized `Remarks` column (its single value already
  # appears in `remarks` with different casing) into `remarks`
  if 'Remarks' in df.columns:
    df['remarks'] = df['remarks'].fillna(df.pop('Remarks'))

  # Files 9 and 10 store the photo reference under different headers
  # (`Image` vs `establishment_pic`); keep one `establishment_pic` column
  if 'establishment_pic' not in df.columns:
    df['establishment_pic'] = pd.NA
  if 'Image' in df.columns:
    df['establishment_pic'] = df['establishment_pic'].fillna(df['Image'])
    df = df.drop(columns='Image')

  return df[columns + ['establishment_pic'] + qa_columns]

In [ ]:
# Parse the mixed-format date_collected column into proper datetime values
def clean_mixed_dates(df):
    """Clean and parse date_collected in place, then return the DataFrame"""

    # Cast to string, strip whitespace, and replace underscores with hyphens
    df['date_collected'] = (
        df['date_collected']
        .astype('string')
        .str.strip()
        .str.replace('_', '-', regex=False)
    )

    # Parse day-first mixed formats; unparseable values are coerced to NaT
    df['date_collected'] = pd.to_datetime(
        df['date_collected'],
        format='mixed',
        dayfirst=True,
        errors='coerce'
    )

    return df

## Importing the raw datas (csv files from Google Drive)

In [ ]:
# Load every raw CSV, clean its dates, tag its source, and collect the results
# uc?export=download&id=

# (Reference) Original Google Drive download URL mapping, now commented out
"""raw_datasets = { # raw datasets and their respective Google Drive URL links dictionary
    'raw_dataset_1': 'https://drive.google.com/uc?export=download&id=162Obps98STe5tsq5bFEl3gAzDPxfqblR',
    'raw_dataset_2': 'https://drive.google.com/uc?export=download&id=1QLZKJY8HiF-IQvFVZbpia9qA2yroEmoS',
    'raw_dataset_3': 'https://drive.google.com/uc?export=download&id=1RXOvapKILTYGc1biL42rjFjlB-I5t9JM',
    'raw_dataset_4': 'https://drive.google.com/uc?export=download&id=1Kze7v7q5nCR0nWUjlPjsjTKUtvWGibDL',
    'raw_dataset_5': 'https://drive.google.com/uc?export=download&id=196zcPBcRJIJgNMhmgRsrgJLwpC5gEZD5',
    'raw_dataset_6': 'https://drive.google.com/uc?export=download&id=1o0KullNbJ5t0cE_fm3QFSagFLfIFmrpC',
    'raw_dataset_7': 'https://drive.google.com/uc?export=download&id=1lTc9XVU5lgTNG0dGgKnnGnw3TGQJjq0S',
    'raw_dataset_8': 'https://drive.google.com/uc?export=download&id=1Celn9JNcsmKjmjjsxSo9BwW0BxSzspn1',
    'raw_dataset_9': 'https://drive.google.com/uc?export=download&id=1-_jGqHAioiHgjfKxUuWmuyV2ebja9Qoc',
    'raw_dataset_10': 'https://drive.google.com/uc?export=download&id=1KEqPvlOY4M43ptcfC2Qmg5CKpDb9ravw'
}"""

raw_datasets = { # raw datasets and their respective Google Drive URL links dictionary
    'raw_dataset_1': 'csv/raw_dataset_1.csv',
    'raw_dataset_2': 'csv/raw_dataset_2.csv',
    'raw_dataset_3': 'csv/raw_dataset_3.csv',
    'raw_dataset_4': 'csv/raw_dataset_4.csv',
    'raw_dataset_5': 'csv/raw_dataset_5.csv',
    'raw_dataset_6': 'csv/raw_dataset_6.csv',
    'raw_dataset_7': 'csv/raw_dataset_7.csv',
    'raw_dataset_8': 'csv/raw_dataset_8.csv',
    'raw_dataset_9': 'csv/raw_dataset_9.csv',
    'raw_dataset_10': 'csv/raw_dataset_10.csv'
}

# Holds the cleaned DataFrame for each raw dataset, keyed by dataset name
processed_datasets = {}

# For each raw CSV: read it, clean its dates, tag it with its source file name, store it
for name, url in raw_datasets.items():
    df = read_csv_files(url)
    df = clean_mixed_dates(df)
    df['source_file'] = name
    processed_datasets[name] = df

### Checking individual datasets

In [ ]:
# Preview the first 5 rows of raw_dataset_1 to spot-check the load
df1 = processed_datasets['raw_dataset_1']
df1.head()

In [ ]:
# Preview the first 5 rows of raw_dataset_2 to spot-check the load
df2 = processed_datasets['raw_dataset_2']
df2.head()

In [ ]:
# Preview the first 5 rows of raw_dataset_3 to spot-check the load
df3 = processed_datasets['raw_dataset_3']
df3.head()

In [ ]:
# Preview the first 5 rows of raw_dataset_4 to spot-check the load
df4 = processed_datasets['raw_dataset_4']
df4.head()

In [ ]:
# Preview the first 5 rows of raw_dataset_5 to spot-check the load
df5 = processed_datasets['raw_dataset_5']
df5.head()

In [ ]:
# Preview the first 5 rows of raw_dataset_6 to spot-check the load
df6 = processed_datasets['raw_dataset_6']
df6.head()

In [ ]:
# Preview the first 5 rows of raw_dataset_7 to spot-check the load
df7 = processed_datasets['raw_dataset_7']
df7.head()

In [ ]:
# Preview the first 5 rows of raw_dataset_8 to spot-check the load
df8 = processed_datasets['raw_dataset_8']
df8.head()

In [ ]:
# Preview the first 5 rows of raw_dataset_9 to spot-check the load
df9 = processed_datasets['raw_dataset_9']
df9.head()

In [ ]:
# Preview the first 5 rows of raw_dataset_10 to spot-check the load
df10 = processed_datasets['raw_dataset_10']
df10.head()

## Setting up the dataframe

In [ ]:
# Data Aggregation

# Stack all 10 cleaned DataFrames into a single raw master DataFrame
df_raw = pd.concat(processed_datasets.values(), ignore_index=True)

# Enforce the spec column order on the merged raw data
df_raw = df_raw[output_columns]

# Work on a copy so the aggregated raw data (df_raw) stays untouched
df = df_raw.copy()

In [ ]:
# Verify all 16 required columns are present, in the spec's order
required_columns = [
    'feature_id', 'name', 'category', 'subcategory',
    'street', 'barangay', 'latitude', 'longitude',
    'location_description', 'operating_status', 'opening_hours',
    'contact_info', 'date_collected', 'collector',
    'remarks', 'verified'
]

missing_columns = [c for c in required_columns if c not in df_raw.columns]
assert not missing_columns, f"Missing required columns: {missing_columns}"

assert list(df_raw.columns[:len(required_columns)]) == required_columns, (
    "Required columns are out of spec order"
)

print("OK:", len(required_columns), "required columns present, in spec order")
print("QA/traceability columns:", list(df_raw.columns[len(required_columns):]))

In [ ]:
# Structural audit: column dtypes, non-null counts, and memory usage
df.info()

In [ ]:
# Summary audit: shape, missing values, duplicate rows, and duplicate feature IDs
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nUnique feature IDs:")
print(df['feature_id'].nunique())

print("\nDuplicate feature IDs:")
print(df['feature_id'].duplicated().sum())

In [ ]:
# Statistical audit: descriptive statistics per column (transposed for readability)
df.describe().T

In [ ]:
# Frequency of each feature_id to reveal duplicated or repeated records
df['feature_id'].value_counts()

In [ ]:
# Compare unique feature IDs against total entries to gauge duplication
print("feature_id:", df['feature_id'].nunique(), "\ntotal entries:", len(df))

In [ ]:
# Frequency of establishment names to spot possible repeat entries
df['name'].value_counts()

In [ ]:
# Distribution of records across barangays
df['barangay'].value_counts()

In [ ]:
# Top 15 data collectors by number of records
df['collector'].value_counts().head(15)

In [ ]:
# Distribution of category values (first pass)
df['category'].value_counts()

In [ ]:
# Distribution of category values (repeat of the previous cell)
df['category'].value_counts()

In [ ]:
# Distribution of subcategory values
df['subcategory'].value_counts()

In [ ]:
# Distribution of operating_status values to expose casing and typo variants
df['operating_status'].value_counts()

In [ ]:
# Top 20 streets by number of records
df['street'].value_counts().head(20)

In [ ]:
# Count of verification flags across the dataset
df['verified'].value_counts()

In [ ]:
# Most frequent latitude values to spot repeated or default coordinates
df['latitude'].value_counts().head(10)

In [ ]:
# Most frequent longitude values to spot repeated or default coordinates
df['longitude'].value_counts().head(10)

In [ ]:
# Count rows that share identical (latitude, longitude) pairs
# (compare as numeric values so differently formatted strings still match)
coords_numeric = pd.DataFrame({
    'latitude': pd.to_numeric(df['latitude'], errors='coerce'),
    'longitude': pd.to_numeric(df['longitude'], errors='coerce')
})

coords_numeric.duplicated().value_counts()

In [ ]:
# Inspect every record assigned to 'Barangay 1'
df[df['barangay'] == 'Barangay 1']

## Gerenal Data Cleaning and Handling Missing Values

The Structure Audit: See data types and missing value counts

df.info()

The Statistical Audit: See averages, min, max (spots wild outliers instantly)

df.describe()

The Emptiness Audit: Count exactly how many NaN values sit in each column

df.isnull().sum()

The Duplication Audit: Check if rows are accidentally duplicated

df.duplicated().sum()

In [ ]:
# .between() can be used for datetime to check if the dates are within the certain range

In [ ]:
# Rank columns by number of missing values (worst first)
missing = df.isnull().sum().to_frame('n_missing')
missing.sort_values('n_missing', ascending=False)

In [ ]:
# Isolate every row whose feature_id appears more than once
duplicate_ids = df[df['feature_id'].duplicated(keep=False)]

# Sort so each duplicate group sits together for side-by-side comparison
duplicate_ids = duplicate_ids.sort_values('feature_id')

duplicate_ids

In [ ]:
# Isolate non-null duplicated names, sorted by name, to find repeat establishments
duplicate_names = df[
    df['name'].notna() &
    df['name'].duplicated(keep=False)
].sort_values('name')

duplicate_names

In [ ]:
# Isolate rows with duplicate coordinate pairs, sorted by coordinates
# (match on numeric values, display the original coordinate strings)
coords_numeric = pd.DataFrame({
    'latitude': pd.to_numeric(df['latitude'], errors='coerce'),
    'longitude': pd.to_numeric(df['longitude'], errors='coerce')
})

duplicate_coordinates = df[
    coords_numeric.duplicated(keep=False)
].sort_values(['latitude', 'longitude'])

duplicate_coordinates

In [ ]:
# Print latitude/longitude ranges to spot coordinates outside expected bounds
# (coordinates are read as text at load, so cast to numeric before comparing)
lat = pd.to_numeric(df['latitude'], errors='coerce')
lon = pd.to_numeric(df['longitude'], errors='coerce')

print(lat.min())
print(lat.max())

print(lon.min())
print(lon.max())

In [ ]:
# Filter rows with missing or out-of-range coordinates (lat 0-90, lon -180 to 180)
lat = pd.to_numeric(df['latitude'], errors='coerce')
lon = pd.to_numeric(df['longitude'], errors='coerce')

bad_coordinates = df[
    lat.isna() |
    lon.isna() |
    (lat < 0) |
    (lat > 90) |
    (lon < -180) |
    (lon > 180)
]

bad_coordinates

In [ ]:
# Correct the misspelled category label 'Professional Sevices'
category_fixes = {
    'Professional Sevices': 'Professional Services'
}

df['category'] = df['category'].replace(category_fixes)
df['category']

In [ ]:
# Normalize operating_status casing and typos into canonical 'Open'/'Closed'
status_fixes = {
    'open': 'Open',
    'OPEN': 'Open',
    'Oepn': 'Open',
    'close': 'Closed',
    'Close': 'Closed',
    'CLOSED': 'Closed',
    'closed': 'Closed',
    'closel': 'Closed'
}

df['operating_status'] = df['operating_status'].replace(status_fixes)
df['operating_status']

In [ ]:
# Normalize 'Sari-Sari Store' spelling variants into one canonical label
subcategory_fixes = {
    'Sari Sari Store': 'Sari-Sari Store',
    'Sari-sari Store': 'Sari-Sari Store'
}

df['subcategory'] = df['subcategory'].replace(subcategory_fixes)

In [ ]:
# Fix malformed feature_id values (wrong character, hyphen, or stray space)
feature_id_fixes = {
    'LUC_BO5_001': 'LUC_B05_001',
    'LUC-B08_036': 'LUC_B08_036',
    'LUC _B04_008': 'LUC_B04_008'
}

df['feature_id'] = df['feature_id'].replace(feature_id_fixes)

In [ ]:
# Extract the barangay number encoded in feature_id (pattern LUC_B<NN>_)
df['id_barangay'] = (
    df['feature_id']
    .str.extract(r'LUC_B(\d{2})_')[0]
)

In [ ]:
# Convert the extracted barangay code to numeric; failures become NaN
df['id_barangay'] = pd.to_numeric(
    df['id_barangay'],
    errors='coerce'
)

In [ ]:
# Extract the barangay number written in the barangay column itself
df['actual_barangay'] = (
    df['barangay']
    .str.extract(r'(\d+)')[0]
)

# Convert to numeric; unparseable values become NaN
df['actual_barangay'] = pd.to_numeric(
    df['actual_barangay'],
    errors='coerce'
)

In [ ]:
# Flag rows where the feature_id barangay code disagrees with the barangay column
id_barangay_mismatch = df[
    df['id_barangay'] != df['actual_barangay']
]

id_barangay_mismatch

In [ ]:
# Drop the temporary validation columns after the consistency check
df = df.drop(columns=['id_barangay', 'actual_barangay'])

## Manual Corrections and Value Fixing

In [ ]:
# Missing-value report: count and percentage per column, sorted by severity
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
})

# Order columns from most missing to least missing
missing = missing.sort_values(
    'missing_percent',
    ascending=False
)

missing

In [ ]:
# Re-parse date_collected as datetime (safety pass after concatenation)
df['date_collected'] = pd.to_datetime(
    df['date_collected'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

In [ ]:
# Check date_collected range and flag implausible future dates
df['date_collected'].min()
df['date_collected'].max()

# Flag any collection date after 2026-12-31 as implausible
future_dates = df[
    df['date_collected'] > pd.Timestamp('2026-12-31')
]

future_dates

In [ ]:
# Inspect rows whose dates failed to parse (NaT after coercion)
df[df['date_collected'].isna()]